# Phase 4 CJPE - Knowledge distillation: Model 3 (teacher) -> Model 4 (student), LoRMA+ and attention pooling across chunks
Run the cells top to bottom. **First run once with `SMOKE = True`** (~5 min) to verify paths / tokenizer / both checkpoints, then set it to `False`.

**Attach as Kaggle inputs**
1. Model-4 pre-training output (`best.pt` + `best_meta.json`) - the **student** backbone
2. Model-3 pre-training output (`best.pt` + `best_meta.json`) - the **teacher** backbone
3. Output of the `phase4-m3-lorma` run (`ckpt/best_adapter_m3.pt`) - the **fine-tuned teacher** (LoRMA+ adapters + attention-pooling head)
4. The tokenizer files both models were trained with (`vocab_50k.json` + `merges_50k.json`). Kaggle secret `HF` = Hugging Face token.
(Optional: the teacher run's output also lets this notebook reuse its tokenised-data cache.)

**Pipeline**
```
CELL 6 (once, resumable)  teacher reads every train doc (<=16 overlapping windows) -> stores doc logits + 512-d doc representation + dev probs, then leaves the GPU
CELL 7                    student = frozen Model-4 (local-128/256 + memory layers) + LoRMA+ adapters on every projection -> window features
                          -> gated-attention pooling across 1024-token chunks (+ position embeddings, tail branch, per-chunk aux loss) -> binary head
                          loss = (1-a)*CE(label, class-weighted) + a*T^2*KL(teacher||student) + w*(1-cos(proj(student_repr), teacher_repr)) + aux
CELL 8/9                  leak-free dev metrics, teacher-vs-student table, ablations, attention analysis, efficiency, prediction.json
```
`KD_ALPHA = 0, FEAT_W = 0` gives the no-KD baseline (same student, same adapters) for the ablation table.

In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
import re, gc, sys, json, math, time, random, shutil, zipfile, hashlib, warnings
from pathlib import Path
from dataclasses import dataclass, fields
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
warnings.filterwarnings("ignore")
T0 = time.time()

# ------------------------------------------------------------------ USER SETTINGS
SMOKE            = False          # True = ~5 min dry run on a tiny subset. RUN THIS ONCE FIRST.
MODE             = "train_predict"  # "train_predict" | "predict_only" (needs best_adapter_<RUN_TAG>.pt from an earlier run)
TRAIN_BUDGET_MIN = 330            # total student-training minutes (LR schedule is time-based -> always anneals to the end)
MAX_EPOCHS       = 3
SESSION_LIMIT_H  = 11.0           # stop training in time to still run inference before Kaggle's 12h wall
RESERVE_MIN      = 25             # minutes kept free for dev + test inference

# optional explicit paths (None = auto-search /kaggle/input)
TEACHER_BACKBONE_PATH = None      # Model-3 pre-trained best.pt
TEACHER_ADAPTER_PATH  = None      # best_adapter_m3.pt written by the phase4-m3-lorma notebook (the FINE-TUNED teacher)
STUDENT_CKPT_PATH     = None      # Model-4 pre-trained best.pt
TOKENIZER_DIR         = None      # folder holding vocab_50k.json + merges_50k.json

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE4_FILES = {"train": "Phase_4_CJPE/Train/train.json",
                "dev":   "Phase_4_CJPE/Train/dev.json",
                "test":  "Phase_4_CJPE/Test/test.json"}
LOWERCASE = True                  # same as the teacher run (dev/test are lowercased)
EXCLUDE_TEST_IDS_FROM_DEV = True  # every test ID also sits in dev.json; keep them out of model selection / threshold tuning
MAX_CHARS, HEAD_CHARS, TAIL_CHARS = 400_000, 80_000, 240_000

# overlapping windows (geometry both models were pretrained with: 1024 ctx, stride 768)
CHUNK, STRIDE = 1024, 768
K_TRAIN, K_QUICK, MAX_INFER_WINDOWS = 6, 8, 40
NB = 8                            # relative-position buckets (+ first + last)

KD_ALPHA      = 0.6               # loss = (1-a)*CE(hard label) + a*T^2*KL(teacher||student) + FEAT_W*cosine-feature-KD + AUX_W*chunk-aux
KD_TEMP       = 2.0
FEAT_W        = 0.5               # doc-representation distillation ([attn-pooled ; tail] vector) through a learned projector
K_TEACHER     = 16                # windows per train doc the teacher looks at (inference-style coverage)
TEACHER_BLOCK = 128               # docs per teacher block (also the partial-save granularity)
TEACHER_DEV   = True              # also store teacher dev probabilities (teacher-vs-student table, ensemble check)

# ---- student adapter: LoRMA+ (default, the phase-4 requirement) or plain LoRA
ADAPTER = "lorma_plus"            # "lorma_plus" (I + s*BA applied on W0 x) | "lora" (additive baseline)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
STUDENT_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "wq_proj", "wk_proj", "wv_proj", "write_out",
                   "gate_proj", "w_gate", "w_up", "w_down")

# optimisation
MICRO_DOCS, ACCUM = 2, 8          # 2 docs x 6 windows per micro-batch; 16 docs / optimizer step
LR_LORA, LR_HEAD, WD_HEAD = 3e-4, 1e-3, 0.01
WARMUP_STEPS, MIN_LR_RATIO, GRAD_CLIP = 50, 0.05, 1.0
AUX_W = 0.3                       # weight of the per-chunk auxiliary loss
LOG_EVERY, EVAL_EVERY_MIN, CKPT_EVERY_MIN = 20, 30.0, 15.0
N_QUICK = 400
INFER_BS, DOC_BS = 32, 64
SEED = 1234

if SMOKE:
    TRAIN_BUDGET_MIN, MAX_EPOCHS, RESERVE_MIN = 3, 1, 0
    EVAL_EVERY_MIN, CKPT_EVERY_MIN, N_QUICK, MAX_INFER_WINDOWS, K_TEACHER = 1.0, 1.0, 40, 8, 8
    N_SMOKE = {"train": 160, "dev": 90, "test": 40}

USE_KD = KD_ALPHA > 0 or FEAT_W > 0
RUN_TAG = "m4kd" if USE_KD else "m4_nokd"

# ------------------------------------------------------------------ paths / device
IS_KAGGLE  = os.path.isdir("/kaggle/working")
INPUT_ROOT = "/kaggle/input"
WORK = Path("/kaggle/working" if IS_KAGGLE else ".") / ("phase4_m4_kd" + ("_smoke" if SMOKE else ""))
CACHE, CKPT, OUT, SUB = WORK / "cache", WORK / "ckpt", WORK / "out", WORK / "submission"
for _d in (CACHE, CKPT, OUT, SUB):
    _d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
OOM_ERR = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)

def amp_ctx(enabled=None):
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP if enabled is None else enabled)

def log(*a):
    print(*a, flush=True)

def fmt_t(sec):
    sec = int(max(sec, 0)); return f"{sec // 3600}h{(sec % 3600) // 60:02d}m"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(SEED)

def read_json(path):
    try:
        with open(path, encoding="utf-8") as f: return json.load(f)
    except Exception:
        return None

def find_files(name, roots):
    hits = []
    for r in roots:
        if r and os.path.isdir(r):
            for d, _, fs in os.walk(r):
                if name in fs: hits.append(os.path.join(d, name))
    return hits

def atomic_save(obj, path):
    tmp = str(path) + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

def _hf_token():
    t = os.environ.get("HF_TOKEN")
    if t: return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")      # never hard-code the token in a notebook
    except Exception:
        return None

def _download(split):
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE4_FILES[split], repo_type="dataset", token=_hf_token())

def _records(split):
    obj = read_json(_download(split))
    if isinstance(obj, dict):
        for k in ("data", "records", "examples"):
            if isinstance(obj.get(k), list): obj = obj[k]; break
    return obj

_ZW, _CTRL = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff]"), re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
_HYPH, _WRAP = re.compile(r"(?<=\w)-\n(?=\w)"), re.compile(r"(?<=\w)\n(?=\w)")
_SP, _NLSP, _NL3 = re.compile(r"[ \t]+"), re.compile(r"[ \t]*\n[ \t]*"), re.compile(r"\n{3,}")

def prep_text(t):
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    t = _CTRL.sub(" ", _ZW.sub("", t)).replace("\ufffd", " ")
    t = _HYPH.sub("", t)
    t = _WRAP.sub(" ", t)                      # undo hard line-wraps (dev/test are wrapped at ~50 chars)
    t = _NL3.sub("\n\n", _NLSP.sub("\n", _SP.sub(" ", t))).strip()
    if len(t) > MAX_CHARS:                     # keep the beginning and (mostly) the end of giant judgments
        t = t[:HEAD_CHARS] + "\n" + t[-TAIL_CHARS:]
    return t.lower() if LOWERCASE else t

log(f"device={DEVICE} | amp={USE_AMP} ({AMP_DTYPE}) | SMOKE={SMOKE} | MODE={MODE} | adapter={ADAPTER} | KD={USE_KD} (alpha={KD_ALPHA}, T={KD_TEMP}, feat={FEAT_W}) | run={RUN_TAG} | work dir={WORK}")
if DEVICE.type == "cuda":
    log("GPU:", torch.cuda.get_device_name(0), f"| {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")

device=cuda | amp=True (torch.float16) | SMOKE=False | MODE=train_predict | adapter=lorma_plus | KD=True (alpha=0.6, T=2.0, feat=0.5) | run=m4kd | work dir=/kaggle/working/phase4_m4_kd
GPU: Tesla T4 | 14.6 GB


In [2]:
#   + LoRMA+ / LoRA adapters, adapter merging, gated-attention pooling head, KD loss terms
# State-dict keys / shapes are identical to the originals, so the checkpoints load with strict=True.
# ---------------- shared helpers ----------------
def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()

def repeat_kv(x, groups):
    if groups == 1: return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)

# ---------------- Model 3 (TEACHER) ----------------
@dataclass
class M3Config:
    vocab_size: int = 50048
    real_vocab: int = 50000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 1536
    dropout: float = 0.1
    max_len: int = 1088
    mtp_heads: int = 3

class M3RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        inv_rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)
        return x * inv_rms * self.weight.to(x.dtype)

def m3_apply_rope(x, cos, sin):
    cos, sin = cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)

class M3GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.kv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)
    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = m3_apply_rope(q, cos, sin), m3_apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

class M3SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up   = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class M3Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.attn = M3RMSNorm(cfg.d_model), M3GQA(cfg)
        self.norm2, self.ffn = M3RMSNorm(cfg.d_model), M3SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        return x + self.drop(self.ffn(self.norm2(x)))

class M3MTPHead(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.proj = nn.Linear(dim, dim, bias=False); self.norm = M3RMSNorm(dim)
    def forward(self, x):
        return self.norm(x + self.proj(x))

class M3DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([M3Block(cfg) for _ in range(cfg.n_layers)])
        self.heads = nn.ModuleList([M3MTPHead(cfg.d_model) for _ in range(cfg.mtp_heads)])
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def hidden(self, idx):
        T = idx.size(1)
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        x = self.drop(self.embed(idx))
        for blk in self.blocks: x = blk(x, cos, sin)
        return x

# ---------------- Model 4 (STUDENT): MSM transformer ----------------
@dataclass
class M4Config:
    vocab_size: int = 50000
    d_model: int = 512
    n_heads: int = 8
    n_kv_heads: int = 2
    ffn_hidden: int = 1792
    attn_dropout: float = 0.0
    ffn_dropout: float = 0.10
    layer_types: tuple = ("local_small", "local_large", "memory") * 3
    window_small: int = 128
    window_large: int = 256
    n_mem_chunks: int = 8
    chunk_size: int = 128
    mem_per_chunk: int = 4
    max_len: int = 1088
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2
    n_mtp: int = 3
    mtp_weights: tuple = (1.0, 0.5, 0.25)
    mtp_pos_frac: float = 0.25
    @property
    def n_layers(self):
        return len(self.layer_types)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)

def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

LOCAL_IMPL = {"mode": "block"}     # "block" = block-wise sliding window (fast, exact); startup self-test falls back to "mask"
_BLOCK_MASKS = {}

def block_local_mask(window, n_blocks, device):
    key = (window, n_blocks, str(device))
    m = _BLOCK_MASKS.get(key)
    if m is None:
        i = torch.arange(window, device=device)[:, None]
        j = torch.arange(2 * window, device=device)[None, :]
        base = (j > i) & (j <= i + window)
        m = base[None].repeat(n_blocks, 1, 1)
        m[0] &= (j >= window)
        m = m[:, None]
        _BLOCK_MASKS[key] = m
    return m

def local_attention(q, k, v, window, dropout_p=0.0):
    B, H, T, hd = q.shape
    nb = (T + window - 1) // window
    Tp = nb * window
    if Tp != T:
        q, k, v = (F.pad(t, (0, 0, 0, Tp - T)) for t in (q, k, v))
    qb = q.reshape(B, H, nb, window, hd).transpose(1, 2).reshape(B * nb, H, window, hd)
    def two_blocks(t):
        tb = t.reshape(B, H, nb, window, hd)
        prev = torch.cat([tb.new_zeros(B, H, 1, window, hd), tb[:, :, :-1]], dim=2)
        return torch.cat([prev, tb], dim=3).transpose(1, 2).reshape(B * nb, H, 2 * window, hd)
    mask = block_local_mask(window, nb, q.device)
    mask = mask[None].expand(B, nb, 1, window, 2 * window).reshape(B * nb, 1, window, 2 * window)
    out = F.scaled_dot_product_attention(qb, two_blocks(k), two_blocks(v), attn_mask=mask, dropout_p=dropout_p)
    return out.reshape(B, nb, H, window, hd).transpose(1, 2).reshape(B, H, Tp, hd)[:, :, :T]

def local_attention_ref(q, k, v, window, dropout_p=0.0):
    T = q.size(2)
    i = torch.arange(T, device=q.device)
    mask = (i[None, :] <= i[:, None]) & (i[None, :] > i[:, None] - window)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dropout_p)

class GQAttention(nn.Module):
    def __init__(self, cfg, window):
        super().__init__()
        self.h, self.kv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.window, self.p = window, cfg.attn_dropout
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)
    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        attend = local_attention if LOCAL_IMPL["mode"] == "block" else local_attention_ref
        out = attend(q, k, v, self.window, self.p if self.training else 0.0)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class LocalBlock(nn.Module):
    kind = "local"
    def __init__(self, cfg, window):
        super().__init__()
        self.norm1, self.attn = RMSNorm(cfg.d_model), GQAttention(cfg, window)
        self.norm2, self.ffn = RMSNorm(cfg.d_model), SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.ffn_dropout)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        return x + self.drop(self.ffn(self.norm2(x)))

def memory_masks(q_chunk, k_chunk, k_valid, mem_per_chunk, n_slots):
    slot_chunk = torch.arange(n_slots, device=q_chunk.device) // mem_per_chunk
    w = (k_chunk[:, None, :] == slot_chunk[None, :, None]) & k_valid[:, None, :]
    w = torch.where(w.any(-1, keepdim=True), w, k_valid[:, None, :])
    r = slot_chunk[None, None, :] < q_chunk[:, :, None]
    has = r.any(-1)
    r = torch.where(has[..., None], r, torch.ones_like(r))
    return w[:, None], r[:, None], has

class MemoryBlock(nn.Module):
    kind = "memory"
    def __init__(self, cfg):
        super().__init__()
        d = cfg.d_model
        self.h, self.kv = cfg.n_heads, cfg.n_kv_heads
        self.hd = d // self.h
        self.groups = self.h // self.kv
        self.p = cfg.attn_dropout
        self.norm1, self.norm2 = RMSNorm(d), RMSNorm(d)
        self.mem_norm_w, self.mem_norm_r = RMSNorm(d), RMSNorm(d)
        self.wq_proj = nn.Linear(d, self.h * self.hd, bias=False)
        self.wk_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.wv_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.write_out = nn.Linear(self.h * self.hd, d, bias=False)
        self.gate_proj = nn.Linear(2 * d, d, bias=False)
        self.q_proj = nn.Linear(d, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(d, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, d, bias=False)
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.ffn_dropout)
    def write_kv(self, xn):
        B, T, _ = xn.shape
        k = self.wk_proj(xn).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.wv_proj(xn).view(B, T, self.kv, self.hd).transpose(1, 2)
        return k, v
    def update(self, mem_prev, k_all, v_all, w_mask):
        B, n, _ = mem_prev.shape
        q = self.wq_proj(self.mem_norm_w(mem_prev)).view(B, n, self.h, self.hd).transpose(1, 2)
        keys, vals = repeat_kv(k_all, self.groups).to(q.dtype), repeat_kv(v_all, self.groups).to(q.dtype)
        out = F.scaled_dot_product_attention(q, keys, vals, attn_mask=w_mask, dropout_p=self.p if self.training else 0.0)
        cand = self.write_out(out.transpose(1, 2).reshape(B, n, -1)).type_as(mem_prev)
        gate = torch.sigmoid(self.gate_proj(torch.cat([mem_prev, cand], dim=-1)))
        return mem_prev + gate * (cand - mem_prev)
    def read(self, xn, mem, r_mask, has):
        B, T, _ = xn.shape
        n = mem.size(1)
        mn = self.mem_norm_r(mem)
        q = self.q_proj(xn).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(mn).view(B, n, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(mn).view(B, n, self.kv, self.hd).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, repeat_kv(k, self.groups), repeat_kv(v, self.groups), attn_mask=r_mask,
                                             dropout_p=self.p if self.training else 0.0)
        out = self.o_proj(out.transpose(1, 2).reshape(B, T, -1))
        return out * has[..., None].to(out.dtype)
    def forward(self, x, mem_prev, mctx):
        xn = self.norm1(x)
        k, v = self.write_kv(xn)
        mem = self.update(mem_prev, k, v, mctx["w"])
        x = x + self.read(xn, mem, mctx["r"], mctx["has"])
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x, mem

class M4DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.n_slots = cfg.n_mem_chunks * cfg.mem_per_chunk
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        windows = {"local_small": cfg.window_small, "local_large": cfg.window_large}
        self.blocks = nn.ModuleList([MemoryBlock(cfg) if k == "memory" else LocalBlock(cfg, windows[k]) for k in cfg.layer_types])
        self.final_norm = RMSNorm(cfg.d_model)
        self.mem_init = nn.Parameter(torch.zeros(self.n_slots, cfg.d_model))
        self.mtp_proj = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False) for _ in range(cfg.n_mtp - 1)])   # only so strict loading works
        self.mtp_norm = nn.ModuleList([RMSNorm(cfg.d_model) for _ in range(cfg.n_mtp - 1)])
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def trunk(self, idx):
        """(B, T) ids -> (B, T, D) final-normed hidden states. Causal everywhere, so RIGHT padding never reaches a real token."""
        B, T = idx.shape
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        x = self.embed(idx)
        mem, mctx = None, None
        for blk in self.blocks:
            if blk.kind == "memory":
                if mem is None:
                    mem = self.mem_init[None].expand(B, -1, -1)
                    pos = torch.arange(T, device=idx.device)[None]
                    chunk = (pos // self.cfg.chunk_size).clamp(max=self.cfg.n_mem_chunks - 1)
                    w, r, has = memory_masks(chunk, chunk, torch.ones_like(pos, dtype=torch.bool), self.cfg.mem_per_chunk, self.n_slots)
                    mctx = {"w": w, "r": r, "has": has}
                x, mem = blk(x, mem, mctx)
            else:
                x = blk(x, cos, sin)
        return self.final_norm(x)

# ---------------- adapters: LoRMA+ (multiplicative) / LoRA (additive) ----------------
class AdaptedLinear(nn.Module):
    """frozen base Linear + trainable low-rank update.
    'lorma_plus': W' = (I + (alpha/r) B A) W0   (Bihany et al., ACL Findings 2025), A: r x d_out, B: d_out x r, computed as y = y0 + s*B(A y0), y0 = W0 x. B = 0 at init.
    'lora'      : y = W0 x + s*B(A x)"""
    def __init__(self, base, r, alpha, p, mode):
        super().__init__()
        self.base, self.mode, self.scale = base, mode, alpha / r
        a_in = base.in_features if mode == "lora" else base.out_features
        self.A = nn.Parameter(torch.empty(r, a_in)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.drop = nn.Dropout(p)
    def forward(self, x):
        y = self.base(x)
        z = x if self.mode == "lora" else y
        return y + F.linear(F.linear(self.drop(z), self.A), self.B) * self.scale

def apply_adapters(m, targets, r, alpha, p, mode):
    """wraps every nn.Linear whose attribute name is in `targets` (works for Model-3 and for Model-4's memory blocks)"""
    n = 0
    for parent in list(m.modules()):
        for name, child in list(parent.named_children()):
            if name in targets and isinstance(child, nn.Linear):
                setattr(parent, name, AdaptedLinear(child, r, alpha, p, mode)); n += 1
    return n

@torch.no_grad()
def merge_adapters(m):
    """fold the adapters into the dense weights (exact in eval mode) -> the teacher runs at plain-backbone speed"""
    for parent in list(m.modules()):
        for name, child in list(parent.named_children()):
            if isinstance(child, AdaptedLinear):
                W0 = child.base.weight.float(); A, B = child.A.float(), child.B.float()
                dW = child.scale * (B @ A) if child.mode == "lora" else child.scale * (B @ (A @ W0))
                child.base.weight.copy_((W0 + dW).to(child.base.weight.dtype)); setattr(parent, name, child.base)

# ---------------- backbone -> window features ----------------
def hidden_states(m, ids):
    return m.trunk(ids) if hasattr(m, "trunk") else m.heads[0](m.hidden(ids))

def backbone_features(m, ids, lens):
    """ids (N,T) right-padded, lens (N,). Per-window feature = [state of the last real token ; mean over real tokens] (2*d_model)."""
    T = ids.size(1)
    h = hidden_states(m, ids).float()
    ar = torch.arange(T, device=ids.device)[None]
    msk = (ar < lens[:, None]).unsqueeze(-1).float()
    last = h[torch.arange(h.size(0), device=ids.device), lens - 1]
    mean = (h * msk).sum(1) / lens[:, None].float()
    return torch.cat([last, mean], dim=-1)

class DocHead(nn.Module):
    """window features -> doc logits. ATTENTION-BASED POOLING ACROSS CHUNKS (gated-attention MIL, Ilse et al. 2018) + chunk-position embeddings
    + tail-chunk branch + per-chunk auxiliary classifier. Same layout as the teacher's head (so its state dict loads); the student additionally
    owns `kd_proj`, the projector that maps its doc representation onto the teacher's."""
    def __init__(self, in_dim, d=256, att=128, p=0.15, kd=False):
        super().__init__()
        self.norm, self.proj = nn.LayerNorm(in_dim), nn.Linear(in_dim, d)
        self.pos = nn.Embedding(NB + 2, d)
        self.drop = nn.Dropout(p)
        self.att_v, self.att_u, self.att_w = nn.Linear(d, att), nn.Linear(d, att), nn.Linear(att, 1)
        self.win_cls = nn.Linear(d, 2)
        self.doc_cls = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, 2))
        self.kd_proj = nn.Linear(2 * d, 2 * d, bias=False) if kd else None
    def forward(self, feats, buckets, mask, return_repr=False):
        h = F.gelu(self.proj(self.drop(self.norm(feats)))) + self.pos(buckets)
        h = self.drop(h)
        a = self.att_w(torch.tanh(self.att_v(h)) * torch.sigmoid(self.att_u(h))).squeeze(-1)
        a = a.masked_fill(~mask, -1e4)
        w = torch.softmax(a, dim=-1) * mask
        pooled = (w.unsqueeze(-1) * h).sum(1)
        last_idx = mask.long().sum(1) - 1
        h_last = h[torch.arange(h.size(0), device=h.device), last_idx]      # windows are ordered => last valid = tail window
        z = torch.cat([pooled, h_last], dim=-1)
        out = (self.doc_cls(z), self.win_cls(h), w)
        return out + (z,) if return_repr else out

def kd_terms(logits, t_logits, s_repr, t_repr, kd_proj):
    """logit distillation (temperature-scaled KL, Hinton et al.) + doc-representation distillation (cosine through a learned projector)"""
    kd = F.kl_div(F.log_softmax(logits / KD_TEMP, -1), F.softmax(t_logits / KD_TEMP, -1), reduction="batchmean") * KD_TEMP ** 2
    ft = 1.0 - F.cosine_similarity(kd_proj(s_repr), t_repr, dim=-1).mean() if kd_proj is not None else logits.new_zeros(())
    return kd, ft

In [3]:
import os
from huggingface_hub import login

# Try retrieving the token from Kaggle Secrets if available
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    # Try getting 'HF_TOKEN' or 'HF' secret
    try:
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        hf_token = user_secrets.get_secret("HF")

    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        login(token=hf_token)
        print("Successfully authenticated with Hugging Face via Kaggle Secrets!")
except Exception as e:
    print("Kaggle secrets not used or failed:", e)
    # Fallback: if running elsewhere, ensure your token is set or login interactively
    # login()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully authenticated with Hugging Face via Kaggle Secrets!


In [4]:
def locate_student():
    if STUDENT_CKPT_PATH: return STUDENT_CKPT_PATH
    best = None
    for h in find_files("best.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "best_meta.json")) or {}
        if "msm" not in str(meta.get("tag", "")): continue            # Model-4 tag = decoder_only_msm_rope_gqa_swiglu_mtp3_v2
        vl = meta.get("val_loss", 1e9)
        if best is None or vl < best[0]: best = (vl, h)
    if best: log(f"Model-4 best.pt: next-token val_loss={best[0]:.4f}"); return best[1]
    for h in find_files("latest.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "latest_meta.json")) or {}
        if "msm" in str(meta.get("tag", "")): log("Model-4 best.pt not found - falling back to latest.pt"); return h
    raise FileNotFoundError("Model-4 (MSM) best.pt / latest.pt not found - attach the Model-4 output dataset or set STUDENT_CKPT_PATH")

def load_student(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = {k.replace("_orig_mod.", "", 1): v for k, v in ck["model"].items()}
    names = {f.name for f in fields(M4Config)}
    kw = {k: v for k, v in dict(ck.get("config", {})).items() if k in names}
    for k in ("layer_types", "mtp_weights"):
        if k in kw: kw[k] = tuple(kw[k])
    cfg = M4Config(**kw)
    cfg.vocab_size, cfg.d_model = sd["embed.weight"].shape
    assert CHUNK <= cfg.max_len, "CHUNK exceeds the RoPE table of the checkpoint"
    if cfg.n_mem_chunks * cfg.chunk_size != CHUNK: log(f"warning: memory chunks tile {cfg.n_mem_chunks * cfg.chunk_size} tokens, windows are {CHUNK}")
    m = M4DecoderLM(cfg)
    m.load_state_dict(sd, strict=True)                    # fails loudly if the architecture copy differs from the pre-training notebook
    log(f"loaded Model-4 weights (pre-training val loss {ck.get('val_loss', float('nan')):.4f}, step {ck.get('global_step')})")
    del ck, sd
    for p in m.parameters(): p.requires_grad = False
    return m.to(DEVICE), cfg

STUDENT_PATH = locate_student()
log(f"student checkpoint: {STUDENT_PATH}")
model, MCFG = load_student(STUDENT_PATH)
log(f"Model-4 backbone: d={MCFG.d_model} layers={MCFG.n_layers} {list(MCFG.layer_types)} heads={MCFG.n_heads}/{MCFG.n_kv_heads}kv ffn={MCFG.ffn_hidden} "
    f"embed_rows={MCFG.vocab_size} | {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")

@torch.no_grad()   # the sliding-window kernel must equal the plain masked reference, else fall back to it
def _local_selftest():
    q, k, v = (torch.randn(2, 4, 300, 16, device=DEVICE) for _ in range(3))
    err = max((local_attention(q, k, v, w) - local_attention_ref(q, k, v, w)).abs().max().item() for w in (MCFG.window_small, MCFG.window_large))
    if err > 1e-4: LOCAL_IMPL["mode"] = "mask"
    log(f"local-attention self-test: max|block - reference| = {err:.2e} -> mode '{LOCAL_IMPL['mode']}'")
_local_selftest()

WORD_PATTERN = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d|[a-zA-Z]+|[0-9]+|[^\s\a-zA-Z0-9]+")

class LegalBPE:
    name = "legalbpe"
    def __init__(self, vocab_path, merges_path):
        self.vocab = read_json(vocab_path)
        merges = read_json(merges_path)
        h = hashlib.md5()
        for p in (vocab_path, merges_path): h.update(open(p, "rb").read())
        self.fingerprint = h.hexdigest()[:10]
        self.ranks = {(a, b): i for i, (a, b) in enumerate(merges)}
        inv = {int(i): t for t, i in self.vocab.items()}
        self.vocab_size = max(inv) + 1
        assert len(inv) == self.vocab_size, "vocab ids are not contiguous"
        self.unk_id = self.vocab.get("<|unk|>", 1)
        self.cache = {}
    def _word_ids(self, word):
        hit = self.cache.get(word)
        if hit is not None: return hit
        syms = list(word) + ["</w>"]
        while len(syms) > 1:
            best, best_rank = None, None
            for i in range(len(syms) - 1):
                r = self.ranks.get((syms[i], syms[i + 1]))
                if r is not None and (best_rank is None or r < best_rank): best, best_rank = (syms[i], syms[i + 1]), r
            if best is None: break
            out, i = [], 0
            while i < len(syms):
                if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]: out.append(best[0] + best[1]); i += 2
                else: out.append(syms[i]); i += 1
            syms = out
        ids = [self.vocab.get(s, self.unk_id) for s in syms]
        if len(self.cache) > 3_000_000: self.cache.clear()
        self.cache[word] = ids
        return ids
    def encode(self, text):
        ids = []
        for w in WORD_PATTERN.findall(text.lower()): ids.extend(self._word_ids(w))
        return ids
    def encode_batch(self, texts):
        return [self.encode(t) for t in texts]

def find_tokenizer():
    for v in find_files("vocab_50k.json", [TOKENIZER_DIR, INPUT_ROOT, str(WORK), os.getcwd()]):
        m = os.path.join(os.path.dirname(v), "merges_50k.json")
        if os.path.exists(m):
            t = LegalBPE(v, m); log(f"tokenizer: {v} (vocab {t.vocab_size})"); return t
    raise FileNotFoundError("vocab_50k.json + merges_50k.json not found - attach the tokenizer files (or set TOKENIZER_DIR)")

TOK = find_tokenizer()
VOCAB = TOK.vocab_size
assert VOCAB <= MCFG.vocab_size, f"tokenizer has {VOCAB} ids but the Model-4 embedding only {MCFG.vocab_size} rows -> wrong tokenizer for this checkpoint"
assert VOCAB < 65536, "token ids are cached as uint16"

@torch.no_grad()
def lm_probe(m, texts, n_tok=512):
    """mean next-token CE of a pre-trained model on CJPE text under the tokenizer (right pair ~3-4 nats, mismatched pair > 7)"""
    m.eval(); tot = cnt = 0
    for t in texts:
        ids = TOK.encode_batch([t])[0][:n_tok + 1]
        if len(ids) < 64: continue
        x = torch.tensor([ids[:-1]], device=DEVICE); y = torch.tensor(ids[1:], device=DEVICE)
        with amp_ctx(): z = hidden_states(m, x)
        tot += F.cross_entropy(F.linear(z.float()[0], m.embed.weight.float()), y, reduction="sum").item(); cnt += y.numel()
    return tot / max(cnt, 1)

_probe = [prep_text(r.get("text") or "")[:12000] for r in _records("dev")[:100]]
PROBE_TEXTS = [t for t in _probe if len(t) > 3000][:16]
probe_s = lm_probe(model, PROBE_TEXTS)
log(f"tokenizer probe: Model-4 next-token loss on {len(PROBE_TEXTS)} CJPE docs = {probe_s:.3f}")
if probe_s > 6.0: log("!! WARNING: very high LM loss -> tokenizer/Model-4 checkpoint pair is probably mismatched. STOP and check the attached inputs.")

# ---------------- teacher builder (called from CELL 6) ----------------
def locate_teacher_backbone():
    if TEACHER_BACKBONE_PATH: return TEACHER_BACKBONE_PATH
    best = None
    for h in find_files("best.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "best_meta.json")) or {}
        tag = str(meta.get("tag", ""))
        if "mtp" not in tag or "msm" in tag: continue                # Model-3 = MTP decoder; Model-4's tag ALSO contains 'mtp' -> exclude 'msm'
        vl = meta.get("val_loss", 1e9)
        if best is None or vl < best[0]: best = (vl, h)
    if best: log(f"Model-3 best.pt: next-token val_loss={best[0]:.4f}"); return best[1]
    for h in find_files("latest.pt", [INPUT_ROOT]):
        meta = read_json(os.path.join(os.path.dirname(h), "latest_meta.json")) or {}
        tag = str(meta.get("tag", ""))
        if "mtp" in tag and "msm" not in tag: log("Model-3 best.pt not found - falling back to latest.pt"); return h
    raise FileNotFoundError("Model-3 best.pt / latest.pt not found - attach the Model-3 checkpoint dataset or set TEACHER_BACKBONE_PATH")

def find_teacher_adapter():
    cands = ([TEACHER_ADAPTER_PATH] if TEACHER_ADAPTER_PATH else []) + find_files("best_adapter_m3.pt", [INPUT_ROOT])
    best = None
    for p in cands:
        try: ck = torch.load(p, map_location="cpu", weights_only=False)
        except Exception as e: log(f"could not read {p}: {e}"); continue
        if ck.get("cfg", {}).get("model") != "model3_mtp": continue
        auc = float(ck.get("quick", {}).get("auc", -1))
        if best is None or auc > best[2]: best = (p, ck, auc)
    if best is None:
        raise FileNotFoundError("fine-tuned teacher not found: attach the OUTPUT of the phase4-m3-lorma run (ckpt/best_adapter_m3.pt) or set TEACHER_ADAPTER_PATH")
    log(f"teacher adapter: {best[0]} (step {best[1].get('step')}, quick-dev AUC {best[2]:.4f})")
    return best[0], best[1]

def build_teacher(ack):
    """Model-3 backbone + its fine-tuned LoRMA+ adapters (merged) + its attention-pooling head. Returns (backbone, head, amp_ok)."""
    tc = ack["cfg"]
    ck = torch.load(locate_teacher_backbone(), map_location="cpu", weights_only=False)
    sd = {k.replace("_orig_mod.", "", 1): v for k, v in ck["model"].items()}
    cd = dict(ck.get("config", {}))
    V, D = sd["embed.weight"].shape
    hd = D // cd["n_heads"] if cd.get("n_heads") else 64
    cfg = M3Config(vocab_size=V, real_vocab=int(cd.get("real_vocab", V)), d_model=D,
                   n_layers=1 + max(int(k.split(".")[1]) for k in sd if k.startswith("blocks.")),
                   n_heads=sd["blocks.0.attn.q_proj.weight"].shape[0] // hd, n_kv_heads=sd["blocks.0.attn.k_proj.weight"].shape[0] // hd,
                   ffn_hidden=sd["blocks.0.ffn.w_gate.weight"].shape[0], dropout=0.1, max_len=int(cd.get("max_len", 1088)),
                   mtp_heads=sum(1 for k in sd if k.startswith("heads.") and k.endswith(".proj.weight")))
    assert tc["feat_dim"] == 2 * cfg.d_model, "teacher adapter was trained on a different backbone width than the attached Model-3 checkpoint"
    tm = M3DecoderLM(cfg); tm.load_state_dict(sd, strict=True); del ck, sd
    for p in tm.parameters(): p.requires_grad = False
    n = apply_adapters(tm, tuple(tc["targets"]), tc["r"], tc["alpha"], 0.0, tc["adapter"])
    own, miss = dict(tm.named_parameters()), 0
    for k, v in ack["lora"].items():
        if k in own: own[k].data.copy_(v)
        else: miss += 1
    assert miss == 0 and len(ack["lora"]) == 2 * n, f"teacher adapter does not match the Model-3 backbone ({miss} unmatched, {len(ack['lora'])} tensors vs {n} adapted layers)"
    th = DocHead(tc["feat_dim"]); th.load_state_dict(ack["head"])
    tm, th = tm.to(DEVICE).eval(), th.to(DEVICE).eval()
    if tc.get("tok_fp") != TOK.fingerprint:
        raise RuntimeError(f"teacher was fine-tuned with tokenizer {tc.get('tokenizer')} (fingerprint {tc.get('tok_fp')}) but this run uses {TOK.name} ({TOK.fingerprint}) -> its targets would be garbage. Attach the same vocab_50k.json + merges_50k.json the teacher run used.")
    # merge + verify the merge is exact, and that autocast is safe for the teacher
    ids = torch.tensor([TOK.encode_batch([PROBE_TEXTS[0]])[0][:CHUNK]], device=DEVICE); lens = torch.tensor([ids.size(1)], device=DEVICE)
    with torch.no_grad(): f_ad = backbone_features(tm, ids, lens)
    merge_adapters(tm)
    with torch.no_grad():
        f_mg = backbone_features(tm, ids, lens)
        with amp_ctx(DEVICE.type == "cuda"): f_16 = backbone_features(tm, ids, lens)
    cos_merge = F.cosine_similarity(f_ad, f_mg, dim=-1).item(); cos_amp = F.cosine_similarity(f_mg, f_16.float(), dim=-1).item()
    amp_ok = DEVICE.type == "cuda" and bool(torch.isfinite(f_16).all()) and cos_amp > 0.98
    log(f"teacher: {sum(p.numel() for p in tm.parameters())/1e6:.1f}M params | merged-vs-adapter cosine {cos_merge:.5f} | fp32-vs-autocast cosine {cos_amp:.5f} -> autocast {'ON' if amp_ok else 'OFF'}")
    assert cos_merge > 0.999, "adapter merge is not exact"
    if lm_probe(tm, PROBE_TEXTS[:8]) > 6.0: log("!! WARNING: Model-3 LM loss is very high under this tokenizer -> mismatched tokenizer/checkpoint")
    return tm, th, amp_ok

Model-4 best.pt: next-token val_loss=2.3764
student checkpoint: /kaggle/input/datasets/krishkalgude/version1/cs781_phase1_model4/checkpoints/best.pt
loaded Model-4 weights (pre-training val loss 2.3764, step 50000)
Model-4 backbone: d=512 layers=9 ['local_small', 'local_large', 'memory', 'local_small', 'local_large', 'memory', 'local_small', 'local_large', 'memory'] heads=8/2kv ffn=1792 embed_rows=50000 | 60.4M params
local-attention self-test: max|block - reference| = 0.00e+00 -> mode 'block'
tokenizer: /kaggle/input/datasets/kalgudekrish/bpe-50k/vocab_50k.json (vocab 50000)


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

tokenizer probe: Model-4 next-token loss on 16 CJPE docs = 3.336


In [5]:
class TokDS:
    """tokenised split stored as one flat uint16 array + offsets"""
    def __init__(self, ids, labels, flat, offs, has_upper):
        self.ids, self.labels, self.flat, self.offs, self.has_upper = list(ids), np.asarray(labels), flat, offs, has_upper
        self.n_tok = np.diff(offs)
    def __len__(self): return len(self.ids)
    def slice(self, i, s, l): return self.flat[self.offs[i] + s: self.offs[i] + s + l]

SIG = f"{'smoke' if SMOKE else 'full'}_lc{int(LOWERCASE)}_ex{int(EXCLUDE_TEST_IDS_FROM_DEV)}_{TOK.name}{TOK.fingerprint}_v{VOCAB}"   # same key as the teacher notebook

def _tokenise(texts, bs=128):
    flat, lens, t0 = [], [], time.time()
    for i in range(0, len(texts), bs):
        for ids in TOK.encode_batch(texts[i:i + bs]):
            a = np.asarray(ids if len(ids) else [0], dtype=np.uint16); flat.append(a); lens.append(len(a))
        if (i // bs) % 40 == 0: log(f"    tokenised {min(i + bs, len(texts)):,}/{len(texts):,} ({time.time()-t0:.0f}s)")
    return np.concatenate(flat), np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)

def build_split(split, records, with_labels):
    t0 = time.time()
    ids = [str(r["id"]) for r in records]
    raw = [r.get("text") or "" for r in records]
    has_upper = np.array([t[:5000] != t[:5000].lower() for t in raw])
    labels = np.array([int(r["label"]) for r in records]) if with_labels else np.full(len(records), -1)   # test labels are NEVER read
    flat, offs = _tokenise([prep_text(t) for t in raw])
    log(f"  {split}: {len(ids):,} docs, {len(flat)/1e6:.1f}M tokens in {(time.time()-t0)/60:.1f} min")
    return TokDS(ids, labels, flat, offs, has_upper)

def save_split(split, ds):
    np.savez(CACHE / f"{split}_{SIG}.npz", ids=np.array(ds.ids), labels=ds.labels, flat=ds.flat, offs=ds.offs, has_upper=ds.has_upper)

def load_split(split):
    name = f"{split}_{SIG}.npz"
    for p in [str(CACHE / name)] + find_files(name, [INPUT_ROOT]):
        if os.path.exists(p):
            try:
                z = np.load(p); return TokDS(z["ids"].tolist(), z["labels"], z["flat"], z["offs"], z["has_upper"])
            except Exception as e:
                log(f"could not read cache {p}: {e}")
    return None

def get_all_splits():
    cached = {s: load_split(s) for s in ("train", "dev", "test")}
    if all(v is not None for v in cached.values()):
        log("tokenised splits loaded from cache"); return cached
    rng = np.random.default_rng(SEED)
    log("reading CJPE json ...")
    test_rec = _records("test")
    test_ids = {str(r["id"]) for r in test_rec}                     # IDs only - test labels are not touched
    test_rec = [{"id": r["id"], "text": r.get("text")} for r in test_rec]
    dev_rec = _records("dev")
    n_overlap = sum(str(r["id"]) in test_ids for r in dev_rec)
    log(f"!! dev/test ID overlap: {n_overlap:,} of {len(test_ids):,} test IDs are also in dev.json")
    if EXCLUDE_TEST_IDS_FROM_DEV:
        dev_rec = [r for r in dev_rec if str(r["id"]) not in test_ids]
        log(f"   -> dev reduced to {len(dev_rec):,} docs that are NOT test cases (used for selection + threshold tuning)")
    tr_rec = _records("train")
    if SMOKE:
        pick_ = lambda recs, n: [recs[i] for i in rng.choice(len(recs), min(n, len(recs)), replace=False)]
        tr_rec, dev_rec, test_rec = pick_(tr_rec, N_SMOKE["train"]), pick_(dev_rec, N_SMOKE["dev"]), pick_(test_rec, N_SMOKE["test"])
    out = {"train": build_split("train", tr_rec, True), "dev": build_split("dev", dev_rec, True), "test": build_split("test", test_rec, False)}
    del tr_rec, dev_rec, test_rec; gc.collect()
    for s, d in out.items(): save_split(s, d)
    return out

SPLITS = get_all_splits()
ds_train, ds_dev, ds_test = SPLITS["train"], SPLITS["dev"], SPLITS["test"]
assert len(set(ds_test.ids)) == len(ds_test.ids), "duplicate test ids"
assert max(int(d.flat.max()) for d in (ds_train, ds_dev, ds_test)) < MCFG.vocab_size, "token id outside the student's embedding table"

# ---------- window geometry ----------
def all_starts(n, W=CHUNK, stride=STRIDE):
    """overlapping window starts; the LAST window is anchored to the document end, so reasoning + disposal are always covered"""
    if n <= W: return np.array([0], dtype=np.int64)
    s = np.arange(0, n - W + 1, stride, dtype=np.int64)
    if s[-1] != n - W: s = np.append(s, n - W)
    return s

def pick(n_win, K, rng=None):
    """coverage-preserving choice of K windows: first + last 2 always, the rest one-per-segment over the middle (random in training, centred at eval)"""
    if n_win <= K: return np.arange(n_win)
    if K == 1: return np.array([n_win - 1])
    if K == 2: return np.array([0, n_win - 1])
    keep = {0, n_win - 2, n_win - 1}
    m = K - 3
    if m > 0:
        lo, hi = 1, n_win - 3
        edges = np.linspace(lo, hi + 1, m + 1)
        for i in range(m):
            a = int(np.ceil(edges[i])); b = max(int(np.ceil(edges[i + 1])), a + 1)
            keep.add(min(int(rng.integers(a, b)) if rng is not None else (a + b - 1) // 2, hi))
    return np.array(sorted(keep))

def bucket_ids(g, n_all):
    """position-embedding id of a window: 0 = first, NB+1 = last, 1..NB = relative position in between"""
    g = np.asarray(g)
    if n_all <= 1: return np.zeros(len(g), dtype=np.int64)
    b = 1 + np.minimum((g / (n_all - 1) * NB).astype(np.int64), NB - 1)
    b = np.where(g == n_all - 1, NB + 1, b)
    return np.where(g == 0, 0, b).astype(np.int64)

def audit(name, ds):
    nt = ds.n_tok; nw = np.array([len(all_starts(n)) for n in nt])
    log(f"{name:10s} docs={len(ds):6,} | tokens mean={nt.mean():7.0f} median={np.median(nt):6.0f} p90={np.percentile(nt,90):7.0f} p99={np.percentile(nt,99):8.0f} | "
        f"windows/doc mean={nw.mean():4.1f} p95={np.percentile(nw,95):3.0f} | {nw.sum():,} windows total")
    if (ds.labels >= 0).all():
        c = np.bincount(ds.labels, minlength=2)
        log(f"{'':10s} labels: class0={c[0]:,} ({100*c[0]/c.sum():.1f}%)  class1={c[1]:,} ({100*c[1]/c.sum():.1f}%)  imbalance={c.max()/max(c.min(),1):.2f}")
log("=" * 100); audit("train", ds_train); audit("dev(clean)", ds_dev); audit("test", ds_test); log("=" * 100)

cnt = np.bincount(ds_train.labels, minlength=2).astype(np.float64)
CLASS_W = torch.tensor(cnt.sum() / (2.0 * cnt), dtype=torch.float32, device=DEVICE)     # "balanced" inverse-frequency weights (same as the teacher)
log(f"class weights (train, inverse-frequency): {CLASS_W.tolist()}")

tokenised splits loaded from cache
train      docs=37,387 | tokens mean=   3543 median=  2419 p90=   7246 p99=   21437 | windows/doc mean= 4.8 p95= 13 | 180,546 windows total
           labels: class0=22,067 (59.0%)  class1=15,320 (41.0%)  imbalance=1.44
dev(clean) docs= 1,988 | tokens mean=   4173 median=  3168 p90=   7619 p99=   22074 | windows/doc mean= 5.6 p95= 14 | 11,136 windows total
           labels: class0=994 (50.0%)  class1=994 (50.0%)  imbalance=1.00
test       docs= 1,517 | tokens mean=   4286 median=  3271 p90=   8005 p99=   20229 | windows/doc mean= 5.8 p95= 15 | 8,731 windows total
class weights (train, inverse-frequency): [0.8471246361732483, 1.2202023267745972]


In [6]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score, confusion_matrix

n_adapted = apply_adapters(model, STUDENT_TARGETS, LORA_R, LORA_ALPHA, LORA_DROPOUT, ADAPTER)
model.to(DEVICE)
FEAT_DIM = 2 * MCFG.d_model
head = DocHead(FEAT_DIM, kd=(FEAT_W > 0)).to(DEVICE)
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
log(f"Model-4 student: {n_total - n_train:,} frozen params | {n_adapted} adapted Linear layers ({ADAPTER}, r={LORA_R}) -> {n_train:,} trainable ({100*n_train/n_total:.2f}%) "
    f"| attention-pooling head {sum(p.numel() for p in head.parameters()):,} params")

def pad_windows(seqs):
    T = max(len(s) for s in seqs)
    arr = np.zeros((len(seqs), T), dtype=np.int64)
    for i, s in enumerate(seqs): arr[i, :len(s)] = s
    return arr, np.array([len(s) for s in seqs], dtype=np.int64)

def make_train_batch(ds, idxs, rng):
    seqs, doc_of, slot_of, rows, ys = [], [], [], [], []
    for b, di in enumerate(idxs):
        n = int(ds.n_tok[di]); starts = all_starts(n); nw = len(starts)
        sel = pick(nw, K_TRAIN, rng)                                 # random middle windows each epoch = augmentation for long docs
        for slot, g in enumerate(sel):
            s = int(starts[g]); seqs.append(ds.slice(di, s, min(CHUNK, n - s))); doc_of.append(b); slot_of.append(slot)
        rows.append(bucket_ids(sel, nw)); ys.append(int(ds.labels[di]))
    ids, lens = pad_windows(seqs)
    B, Km = len(idxs), max(len(r) for r in rows)
    bucket = np.zeros((B, Km), dtype=np.int64); mask = np.zeros((B, Km), dtype=bool)
    for i, r in enumerate(rows): bucket[i, :len(r)] = r; mask[i, :len(r)] = True
    t = lambda a: torch.from_numpy(a).to(DEVICE, non_blocking=True)
    return dict(ids=t(ids), lens=t(lens), doc_of=t(np.array(doc_of)), slot_of=t(np.array(slot_of)), docs=t(np.asarray(idxs, dtype=np.int64)),
                bucket=t(bucket), mask=t(mask), y=t(np.array(ys, dtype=np.int64)))

@torch.no_grad()
def extract_features(m, ds, doc_indices, kcap, bs=None, tag="", amp=None):
    """run a frozen/adapted backbone over the selected overlapping windows of every doc; returns window features + bookkeeping,
    so any doc-level head / section ablation can be evaluated later without touching the GPU again"""
    bs = bs or INFER_BS
    was_training = m.training; m.eval()
    items, ptr, gidx, nall = [], [0], [], []
    for di in doc_indices:
        n = int(ds.n_tok[di]); starts = all_starts(n); nw = len(starts)
        for g in pick(nw, kcap):
            s = int(starts[g]); items.append((int(di), s, min(CHUNK, n - s))); gidx.append(int(g))
        ptr.append(len(items)); nall.append(nw)
    feats = np.zeros((len(items), 2 * m.cfg.d_model), dtype=np.float32)
    t0 = time.time(); nb = (len(items) + bs - 1) // bs
    for bi in range(nb):
        chunk = items[bi * bs:(bi + 1) * bs]
        ids, lens = pad_windows([ds.slice(di, s, l) for di, s, l in chunk])
        with amp_ctx(amp):
            f = backbone_features(m, torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lens).to(DEVICE))
        feats[bi * bs: bi * bs + len(chunk)] = f.float().cpu().numpy()
        if tag and (bi % 100 == 0 or bi == nb - 1) and bi:
            log(f"    [{tag}] batch {bi+1}/{nb} | {(bi+1)*bs/max(time.time()-t0,1e-6):.0f} windows/s (avg)")
    if was_training: m.train()
    return feats, np.array(ptr), np.array(gidx), np.array(nall)

@torch.no_grad()
def doc_forward(hd, feats, ptr, gidx, nall, keep=None, agg="att", collect_att=False, collect_repr=False):
    """P(class=1) / logits / representation per document from cached window features.
    keep(g, n_all)->bool mask restricts WHICH windows the head may see (section ablations); agg='winmean' = mean of window logits (no attention)"""
    hd.eval(); fd = feats.shape[1]; ND = len(ptr) - 1
    probs, logit_out, att = np.zeros(ND), np.zeros((ND, 2), np.float32), []
    rdim = hd.doc_cls[0].in_features; reprs = np.zeros((ND, rdim), np.float32) if collect_repr else None
    for b0 in range(0, ND, DOC_BS):
        docs = list(range(b0, min(ND, b0 + DOC_BS))); rows = []
        for d in docs:
            sel = np.arange(ptr[d], ptr[d + 1])
            if keep is not None:
                m = np.asarray(keep(gidx[sel], int(nall[d])), dtype=bool)
                sel = sel[m] if m.any() else sel[-1:]
            rows.append(sel)
        Km = max(len(r) for r in rows)
        Fb = np.zeros((len(docs), Km, fd), dtype=np.float32); Bk = np.zeros((len(docs), Km), dtype=np.int64); Mk = np.zeros((len(docs), Km), dtype=bool)
        for i, (d, r) in enumerate(zip(docs, rows)):
            Fb[i, :len(r)] = feats[r]; Bk[i, :len(r)] = bucket_ids(gidx[r], int(nall[d])); Mk[i, :len(r)] = True
        Ft, Bt, Mt = (torch.from_numpy(a).to(DEVICE) for a in (Fb, Bk, Mk))
        logits, wlog, w, z = hd(Ft, Bt, Mt, return_repr=True)
        if agg == "winmean": logits = (wlog * Mt.unsqueeze(-1)).sum(1) / Mt.sum(1, keepdim=True)
        logit_out[b0:b0 + len(docs)] = logits.float().cpu().numpy()
        probs[b0:b0 + len(docs)] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
        if collect_repr: reprs[b0:b0 + len(docs)] = z.float().cpu().numpy()
        if collect_att:
            wn = w.cpu().numpy()
            for i, r in enumerate(rows): att.append((Bk[i, :len(r)], wn[i, :len(r)]))
    return dict(probs=probs, logits=logit_out, repr=reprs, att=att)

def doc_probs(hd, feats, ptr, gidx, nall, keep=None, agg="att", collect_att=False):
    o = doc_forward(hd, feats, ptr, gidx, nall, keep, agg, collect_att)
    return (o["probs"], o["att"]) if collect_att else o["probs"]

def macro_f1_np(y, pred):
    f = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        f.append(2 * p * r / (p + r) if p + r else 0.0)
    return float(np.mean(f))

def tune_threshold(y, p, lo=0.20, hi=0.80, step=0.01, smooth=5):
    """threshold maximising Macro-F1 on a smoothed F1 curve (so one lucky threshold on a small dev set is not chosen)"""
    grid = np.arange(lo, hi + 1e-9, step)
    f = np.array([macro_f1_np(y, (p >= t).astype(int)) for t in grid])
    fs = np.convolve(np.pad(f, smooth // 2, mode="edge"), np.ones(smooth) / smooth, mode="valid")
    i = int(np.argmax(fs)); return float(grid[i]), float(f[i]), float(macro_f1_np(y, (p >= 0.5).astype(int)))

def full_metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    P, R, Fm, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    pc, rc, fc, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    try: auc = roc_auc_score(y, p)
    except ValueError: auc = float("nan")
    return dict(thr=thr, macro_f1=Fm, macro_precision=P, macro_recall=R, accuracy=accuracy_score(y, pred), auc=auc,
                p0=pc[0], r0=rc[0], f1_0=fc[0], p1=pc[1], r1=rc[1], f1_1=fc[1], confusion=confusion_matrix(y, pred, labels=[0, 1]).tolist())

_rs = np.random.default_rng(SEED + 1)
QUICK_IDX = np.sort(_rs.choice(len(ds_dev), min(N_QUICK, len(ds_dev)), replace=False))

def quick_eval():
    was_m, was_h = model.training, head.training
    f, ptr, g, na = extract_features(model, ds_dev, QUICK_IDX, K_QUICK)
    p = doc_probs(head, f, ptr, g, na); y = ds_dev.labels[QUICK_IDX]
    thr, f1_tuned, f1_05 = tune_threshold(y, p)
    try: auc = roc_auc_score(y, p)
    except ValueError: auc = float("nan")
    if was_m: model.train()
    if was_h: head.train()
    return dict(auc=auc, f1_05=f1_05, f1_tuned=f1_tuned, thr=thr)

def amp_sanity():
    """autocast vs fp32 on real windows - falls back to fp32 if they disagree"""
    global USE_AMP
    if not USE_AMP: return
    seqs = [ds_train.slice(i, 0, min(CHUNK, ds_train.n_tok[i])) for i in range(min(6, len(ds_train)))]
    ids, lens = pad_windows(seqs); ids, lens = torch.from_numpy(ids).to(DEVICE), torch.from_numpy(lens).to(DEVICE)
    model.eval()
    with torch.no_grad():
        f32 = backbone_features(model, ids, lens)
        with amp_ctx(): f16 = backbone_features(model, ids, lens)
    cos_min = F.cosine_similarity(f32, f16.float(), dim=-1).min().item()
    ok = bool(torch.isfinite(f16).all()) and cos_min > 0.98
    log(f"AMP sanity (student): min cosine(fp32, {AMP_DTYPE})={cos_min:.5f} finite={bool(torch.isfinite(f16).all())} -> {'autocast ON' if ok else 'DISABLED (fp32)'}")
    USE_AMP = ok
amp_sanity()

Model-4 student: 60,364,288 frozen params | 78 adapted Linear layers (lorma_plus, r=16) -> 1,720,320 trainable (2.77%) | attention-pooling head 727,429 params
AMP sanity (student): min cosine(fp32, torch.float16)=1.00000 finite=True -> autocast ON


In [7]:
T_LOGITS = T_REPR = TEACHER_DEV_P = None
T_META = {}

def teacher_stage():
    global T_LOGITS, T_REPR, TEACHER_DEV_P, T_META
    N, Nd = len(ds_train), len(ds_dev)
    lg, rp, dev_p, meta = np.full((N, 2), np.nan, np.float32), np.zeros((N, 512), np.float16), np.full(Nd, np.nan), {}
    try:
        ap, ack = find_teacher_adapter()
    except FileNotFoundError:
        if USE_KD and MODE == "train_predict": raise
        log("no teacher adapter attached (not needed for predict_only / no-KD) - skipping the teacher stage"); return
    tc = ack["cfg"]
    fp = hashlib.md5(f"{tc.get('tok_fp')}|{ack.get('step')}|{tc.get('r')}|{os.path.getsize(ap)}".encode()).hexdigest()[:8]
    name = f"teacher_kd_{SIG}_{fp}.npz"
    for p in [str(CKPT / name)] + find_files(name, [INPUT_ROOT]):
        if os.path.exists(p):
            try:
                z = np.load(p, allow_pickle=False)
                if z["train_ids"].tolist() == ds_train.ids and z["dev_ids"].tolist() == ds_dev.ids:
                    lg, rp, dev_p, meta = z["logits"].copy(), z["repr"].copy(), z["dev_p"].copy(), json.loads(str(z["meta"]))
                    log(f"teacher cache found: {p} ({int((~np.isnan(lg[:, 0])).sum()):,}/{N:,} train docs, dev {'yes' if not np.isnan(dev_p).any() else 'no'})"); break
            except Exception as e:
                log(f"could not read teacher cache {p}: {e}")

    def save_cache():
        tmp = str(CKPT / name) + ".tmp.npz"
        np.savez(tmp, train_ids=np.array(ds_train.ids), logits=lg, repr=rp, dev_ids=np.array(ds_dev.ids), dev_p=dev_p, meta=np.array(json.dumps(meta)))
        os.replace(tmp, CKPT / name)

    todo = np.where(np.isnan(lg[:, 0]))[0] if (USE_KD and MODE == "train_predict") else np.array([], dtype=int)
    need_dev = TEACHER_DEV and np.isnan(dev_p).any()
    if len(todo) or need_dev:
        tm, th, t_amp = build_teacher(ack)
        meta.update(t_params=int(sum(p.numel() for p in tm.parameters())), t_step=ack.get("step"), t_quick_auc=float(ack.get("quick", {}).get("auc", float("nan"))), k_teacher=K_TEACHER)
        t0, last_save, nwin = time.time(), time.time(), 0
        for b0 in range(0, len(todo), TEACHER_BLOCK):
            idx = todo[b0:b0 + TEACHER_BLOCK]
            f, ptr, g, na = extract_features(tm, ds_train, idx, K_TEACHER, amp=t_amp)
            o = doc_forward(th, f, ptr, g, na, collect_repr=True)
            lg[idx], rp[idx] = o["logits"], o["repr"].astype(np.float16); nwin += len(f)
            el = time.time() - t0; done = b0 + len(idx)
            log(f"  teacher targets: {done:,}/{len(todo):,} docs | {nwin/max(el,1e-6):.0f} windows/s | elapsed {fmt_t(el)} | ETA {fmt_t(el/done*(len(todo)-done))}")
            if time.time() - last_save > 480: meta["t_wps"] = nwin / max(el, 1e-6); save_cache(); last_save = time.time()
        if len(todo): meta["t_wps"] = nwin / max(time.time() - t0, 1e-6)
        if need_dev:
            t1 = time.time()
            f, ptr, g, na = extract_features(tm, ds_dev, np.arange(Nd), MAX_INFER_WINDOWS, tag="teacher-dev", amp=t_amp)
            dev_p = doc_probs(th, f, ptr, g, na); meta["t_dev_wps"] = len(f) / max(time.time() - t1, 1e-6)
        save_cache()
        del tm, th, f; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    elif USE_KD and MODE == "train_predict":
        log("teacher cache already complete - the teacher model is not loaded at all")

    if USE_KD and MODE == "train_predict":
        assert not np.isnan(lg).any(), "teacher targets incomplete"
        y = ds_train.labels; pt = torch.softmax(torch.from_numpy(lg), -1)[:, 1].numpy()
        log(f"teacher on its OWN train docs: acc@0.5 {np.mean((pt >= 0.5) == y):.4f} | AUC {roc_auc_score(y, pt):.4f} | mean max-prob {np.maximum(pt, 1 - pt).mean():.3f}"
            f"  (near-perfect = the teacher memorised its train set -> rely on KD_TEMP / (1-KD_ALPHA) to keep the hard labels in play)")
        T_LOGITS = torch.from_numpy(lg).to(DEVICE); T_REPR = torch.from_numpy(rp.astype(np.float32)).to(DEVICE)
    if not np.isnan(dev_p).any():
        TEACHER_DEV_P = dev_p
        log(f"teacher dev: AUC {roc_auc_score(ds_dev.labels, dev_p):.4f} | Macro-F1@0.5 {macro_f1_np(ds_dev.labels, (dev_p >= 0.5).astype(int)):.4f}")
    T_META = meta

if USE_KD or TEACHER_DEV:
    teacher_stage()
else:
    log("teacher stage skipped (no KD, no teacher comparison)")

teacher adapter: /kaggle/input/datasets/krishkalgude215/p4-m3-teacher/phase4_model3_lorma/ckpt/best_adapter_m3.pt (step 5160, quick-dev AUC 0.8956)
Model-3 best.pt: next-token val_loss=2.3525
teacher: 64.7M params | merged-vs-adapter cosine 1.00000 | fp32-vs-autocast cosine 1.00000 -> autocast ON
  teacher targets: 128/37,387 docs | 140 windows/s | elapsed 0h00m | ETA 0h26m
  teacher targets: 256/37,387 docs | 142 windows/s | elapsed 0h00m | ETA 0h25m
  teacher targets: 384/37,387 docs | 142 windows/s | elapsed 0h00m | ETA 0h25m
  teacher targets: 512/37,387 docs | 142 windows/s | elapsed 0h00m | ETA 0h25m
  teacher targets: 640/37,387 docs | 141 windows/s | elapsed 0h00m | ETA 0h24m
  teacher targets: 768/37,387 docs | 141 windows/s | elapsed 0h00m | ETA 0h24m
  teacher targets: 896/37,387 docs | 141 windows/s | elapsed 0h00m | ETA 0h24m
  teacher targets: 1,024/37,387 docs | 141 windows/s | elapsed 0h00m | ETA 0h24m
  teacher targets: 1,152/37,387 docs | 140 windows/s | elapsed 0h00m

In [8]:
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW([{"params": trainable, "lr": LR_LORA, "weight_decay": 0.0},
                         {"params": list(head.parameters()), "lr": LR_HEAD, "weight_decay": WD_HEAD}], betas=(0.9, 0.999))
BASE_LRS = [LR_LORA, LR_HEAD]
_use_scaler = USE_AMP and AMP_DTYPE == torch.float16
try:    scaler = torch.amp.GradScaler("cuda", enabled=_use_scaler)
except Exception: scaler = torch.cuda.amp.GradScaler(enabled=_use_scaler)

S = dict(epoch=0, doc_ptr=0, step=0, train_sec=0.0, docs=0, wins=0, best_auc=-1.0, best_step=-1)
LATEST, BEST = CKPT / f"latest_cjpe_{RUN_TAG}.pt", CKPT / f"best_adapter_{RUN_TAG}.pt"
RUN_CFG = dict(model="model4_msm", run=RUN_TAG, adapter=ADAPTER, r=LORA_R, alpha=LORA_ALPHA, chunk=CHUNK, stride=STRIDE, k_train=K_TRAIN, lowercase=LOWERCASE,
               feat_dim=FEAT_DIM, targets=list(STUDENT_TARGETS), tokenizer=TOK.name, tok_fp=TOK.fingerprint,
               kd=dict(alpha=KD_ALPHA, temp=KD_TEMP, feat_w=FEAT_W, k_teacher=K_TEACHER))
_compat = lambda c: c.get("model") == "model4_msm" and c.get("run") == RUN_TAG and c.get("adapter") == ADAPTER and c.get("feat_dim") == FEAT_DIM and c.get("tok_fp") == TOK.fingerprint

def adapter_state():
    return {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}

def load_adapter_state(lora_sd, head_sd):
    own = dict(model.named_parameters()); miss = 0
    for n, v in lora_sd.items():
        if n in own: own[n].data.copy_(v.to(own[n].device))
        else: miss += 1
    head.load_state_dict(head_sd, strict=False)           # strict=False: a no-KD run has no kd_proj, a resumed KD run does
    if miss: log(f"warning: {miss} adapter tensors in the checkpoint did not match this model")

def save_latest():
    atomic_save({"lora": adapter_state(), "head": head.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "state": dict(S), "cfg": RUN_CFG}, LATEST)

def save_best(res):
    atomic_save({"lora": adapter_state(), "head": head.state_dict(), "cfg": RUN_CFG, "quick": res, "step": S["step"]}, BEST)

def try_resume():
    if SMOKE: return False
    cands = [p for p in [str(LATEST)] + find_files(LATEST.name, [INPUT_ROOT]) if os.path.exists(p)]
    best = None
    for p in cands:
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if _compat(ck.get("cfg", {})) and (best is None or ck["state"]["train_sec"] > best[1]["state"]["train_sec"]): best = (p, ck)
        except Exception as e:
            log(f"could not read {p}: {e}")
    if best is None: return False
    p, ck = best
    load_adapter_state(ck["lora"], ck["head"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"]); S.update(ck["state"])
    log(f"RESUMED from {p}: step {S['step']}, epoch {S['epoch']}, {S['train_sec']/60:.0f} min already trained, best quick AUC {S['best_auc']:.4f}")
    prev_best = os.path.join(os.path.dirname(p), BEST.name)
    if not BEST.exists() and os.path.exists(prev_best): shutil.copy(prev_best, BEST)
    return True

def lr_mult(step, progress):
    warm = min(1.0, (step + 1) / WARMUP_STEPS)
    return warm * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0))))

def micro_step(idxs, rng):
    b = make_train_batch(ds_train, idxs, rng)
    with amp_ctx():
        feats = backbone_features(model, b["ids"], b["lens"])
    feats = feats.float()
    B, Km = b["mask"].shape
    padded = torch.zeros(B, Km, feats.size(-1), device=feats.device).index_put((b["doc_of"], b["slot_of"]), feats)
    logits, wlog, _, srepr = head(padded, b["bucket"], b["mask"], return_repr=True)
    loss_ce = F.cross_entropy(logits, b["y"], weight=CLASS_W)                                   # class imbalance: inverse-frequency weights
    wy = b["y"][:, None].expand(B, Km)[b["mask"]]
    loss_aux = F.cross_entropy(wlog[b["mask"]], wy, weight=CLASS_W, label_smoothing=0.1)        # every chunk sees the doc label (noisy -> smoothed)
    if USE_KD:
        kd, ft = kd_terms(logits, T_LOGITS[b["docs"]], srepr, T_REPR[b["docs"]], head.kd_proj if FEAT_W > 0 else None)
        loss = (1 - KD_ALPHA) * loss_ce + KD_ALPHA * kd + FEAT_W * ft + AUX_W * loss_aux
    else:
        loss = loss_ce + AUX_W * loss_aux
    if not torch.isfinite(loss): return float("nan"), 0
    scaler.scale(loss / ACCUM).backward()
    return float(loss.item()), int(b["ids"].size(0))

def train():
    N = len(ds_train); budget = TRAIN_BUDGET_MIN * 60.0
    session_cap = max(0.0, (SESSION_LIMIT_H * 60 - RESERVE_MIN) * 60 - (time.time() - T0))
    t_sess, sec0 = time.time(), S["train_sec"]
    last_ckpt = last_eval = t_log = time.time(); w_log, losses, n_oom, stop, docs_sess = 0, [], 0, None, 0
    model.train(); head.train()
    log(f"training: budget {TRAIN_BUDGET_MIN} min total ({sec0/60:.0f} done) | this session may use {session_cap/60:.0f} min | "
        f"{MICRO_DOCS*ACCUM} docs/step x {K_TRAIN} windows | {N:,} train docs | KD={USE_KD}")
    while stop is None and S["epoch"] < MAX_EPOCHS:
        perm = np.random.default_rng(SEED + S["epoch"]).permutation(N)
        rng = np.random.default_rng(SEED * 7 + S["epoch"] * 1000 + S["step"])
        while S["doc_ptr"] < N:
            sess = time.time() - t_sess; elapsed = sec0 + sess; S["train_sec"] = elapsed
            progress = max(elapsed / budget, (S["epoch"] * N + S["doc_ptr"]) / (MAX_EPOCHS * N))
            if progress >= 1.0: stop = "budget reached"; break
            if sess >= session_cap: stop = "session limit reached - resume next session"; break
            m = lr_mult(S["step"], progress)
            for g, base in zip(opt.param_groups, BASE_LRS): g["lr"] = base * m
            opt.zero_grad(set_to_none=True); nmb = 0
            for _ in range(ACCUM):
                if S["doc_ptr"] >= N: break
                idxs = perm[S["doc_ptr"]: S["doc_ptr"] + MICRO_DOCS]; S["doc_ptr"] += len(idxs)
                try:
                    l, nw = micro_step(idxs, rng)
                except OOM_ERR:
                    n_oom += 1; opt.zero_grad(set_to_none=True); torch.cuda.empty_cache()
                    log(f"  CUDA OOM #{n_oom} - skipped a micro-batch (lower MICRO_DOCS or K_TRAIN if this repeats)")
                    if n_oom > 5: raise
                    continue
                if nw: losses.append(l); w_log += nw; S["docs"] += len(idxs); S["wins"] += nw; docs_sess += len(idxs); nmb += 1
            if nmb == 0: continue
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable + list(head.parameters()), GRAD_CLIP)
            scaler.step(opt); scaler.update(); S["step"] += 1
            if S["step"] % LOG_EVERY == 0:
                dt = time.time() - t_log; wps = w_log / max(dt, 1e-6)
                eta_epoch = (N - S["doc_ptr"]) / max(docs_sess / max(sess, 1e-6), 1e-6)
                log(f"ep {S['epoch']} step {S['step']:5d} | docs {S['doc_ptr']:6,}/{N:,} | loss {np.mean(losses[-LOG_EVERY*ACCUM:]):.4f} | "
                    f"lr x{m:.2f} | {wps:5.1f} win/s | trained {fmt_t(elapsed)}/{fmt_t(budget)} | epoch ETA {fmt_t(eta_epoch)}")
                t_log, w_log = time.time(), 0
            now = time.time()
            if now - last_eval >= EVAL_EVERY_MIN * 60:
                r = quick_eval(); last_eval = time.time()
                better = (not np.isnan(r["auc"])) and r["auc"] > S["best_auc"]
                if better: S["best_auc"], S["best_step"] = r["auc"], S["step"]; save_best(r)
                log(f"  >> quick-dev ({len(QUICK_IDX)} docs, {K_QUICK} windows): AUC {r['auc']:.4f} | macroF1@0.5 {r['f1_05']:.4f} | "
                    f"macroF1@thr{r['thr']:.2f} {r['f1_tuned']:.4f} {'(NEW BEST)' if better else ''}")
                t_log = time.time()
            if now - last_ckpt >= CKPT_EVERY_MIN * 60: save_latest(); last_ckpt = now
        if stop is None: S["epoch"] += 1; S["doc_ptr"] = 0
    S["train_sec"] = sec0 + (time.time() - t_sess)
    r = quick_eval()
    better = (not np.isnan(r["auc"])) and r["auc"] > S["best_auc"]
    if better or not BEST.exists(): S["best_auc"], S["best_step"] = r["auc"], S["step"]; save_best(r)
    log(f"  >> final quick-dev: AUC {r['auc']:.4f} | macroF1@0.5 {r['f1_05']:.4f} | macroF1@thr{r['thr']:.2f} {r['f1_tuned']:.4f} {'(NEW BEST)' if better else ''}")
    save_latest()
    log(f"training stopped: {stop or 'epochs finished'} | steps {S['step']} | docs seen {S['docs']:,} | windows {S['wins']:,} | best quick AUC {S['best_auc']:.4f} @ step {S['best_step']}")

if MODE == "train_predict":
    try_resume()
    train()
else:
    log("MODE=predict_only: skipping training")

training: budget 330 min total (0 done) | this session may use 610 min | 16 docs/step x 6 windows | 37,387 train docs | KD=True
ep 0 step    20 | docs    320/37,387 | loss 1.3765 | lr x0.40 |  30.9 win/s | trained 0h00m/5h30m | epoch ETA 1h08m
ep 0 step    40 | docs    640/37,387 | loss 1.1812 | lr x0.80 |  31.1 win/s | trained 0h01m/5h30m | epoch ETA 1h08m
ep 0 step    60 | docs    960/37,387 | loss 1.0351 | lr x1.00 |  31.3 win/s | trained 0h01m/5h30m | epoch ETA 1h06m
ep 0 step    80 | docs  1,280/37,387 | loss 1.0196 | lr x1.00 |  31.1 win/s | trained 0h02m/5h30m | epoch ETA 1h06m
ep 0 step   100 | docs  1,600/37,387 | loss 0.9209 | lr x1.00 |  31.6 win/s | trained 0h02m/5h30m | epoch ETA 1h06m
ep 0 step   120 | docs  1,920/37,387 | loss 0.8749 | lr x1.00 |  31.3 win/s | trained 0h03m/5h30m | epoch ETA 1h05m
ep 0 step   140 | docs  2,240/37,387 | loss 0.9296 | lr x1.00 |  31.1 win/s | trained 0h04m/5h30m | epoch ETA 1h04m
ep 0 step   160 | docs  2,560/37,387 | loss 0.8926 | lr x1.0

In [9]:
# (dev = dev.json minus every test-case ID, so nothing here touches test labels)
_best = None
for p in [str(BEST)] + find_files(BEST.name, [INPUT_ROOT]):
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False)
            if _compat(ck["cfg"]): _best = (p, ck); break
        except Exception as e:
            log(f"could not read {p}: {e}")
if _best:
    load_adapter_state(_best[1]["lora"], _best[1]["head"])
    log(f"evaluating best adapter from {_best[0]} (step {_best[1].get('step')}, quick AUC {_best[1]['quick']['auc']:.4f})")
else:
    assert MODE == "train_predict", f"predict_only needs {BEST.name} (attach the training run's output as an input)"
    log("no best adapter found - using the weights currently in memory")

if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
t0 = time.time()
dev_idx = np.arange(len(ds_dev))
f_dev, ptr_dev, g_dev, n_dev = extract_features(model, ds_dev, dev_idx, MAX_INFER_WINDOWS, tag="dev")
dev_sec = time.time() - t0
log(f"dev feature extraction (student): {dev_sec/60:.1f} min")
np.savez(OUT / f"dev_window_feats_{RUN_TAG}.npz", feats=f_dev, ptr=ptr_dev, gidx=g_dev, nall=n_dev, y=ds_dev.labels)
y_dev = ds_dev.labels
p_dev = doc_probs(head, f_dev, ptr_dev, g_dev, n_dev)

THR, f1_tuned, f1_05 = tune_threshold(y_dev, p_dev)
if abs(THR - 0.5) > 0.15: log(f"warning: tuned threshold {THR:.2f} is far from 0.5 - check for train/dev shift before trusting it")
M05, MT = full_metrics(y_dev, p_dev, 0.5), full_metrics(y_dev, p_dev, THR)
log("\n" + "=" * 90 + f"\nSTUDENT DEV RESULTS ({len(y_dev):,} docs, class balance {np.bincount(y_dev, minlength=2).tolist()})\n" + "=" * 90)
for name, m in (("threshold 0.50", M05), (f"tuned thr {THR:.2f}", MT)):
    log(f"{name:15s} | Macro-F1 {m['macro_f1']:.4f} | Macro-P {m['macro_precision']:.4f} | Macro-R {m['macro_recall']:.4f} | Acc {m['accuracy']:.4f} | AUC {m['auc']:.4f}")
    log(f"{'':15s} | class0 P/R/F1 {m['p0']:.3f}/{m['r0']:.3f}/{m['f1_0']:.3f} | class1 P/R/F1 {m['p1']:.3f}/{m['r1']:.3f}/{m['f1_1']:.3f} | confusion [[TN,FP],[FN,TP]] = {m['confusion']}")

# ---------- teacher vs student (the point of distillation) ----------
cmp_rows = []
if TEACHER_DEV_P is not None:
    tp = TEACHER_DEV_P; thr_t, _, _ = tune_threshold(y_dev, tp)
    ens = 0.5 * (tp + p_dev); thr_e, _, _ = tune_threshold(y_dev, ens)
    for nm, pr, th_ in (("teacher  Model-3 + LoRMA+", tp, thr_t), (f"student  Model-4 + LoRMA+ ({RUN_TAG})", p_dev, THR), ("average of both", ens, thr_e)):
        m = full_metrics(y_dev, pr, th_)
        cmp_rows.append(dict(model=nm, AUC=m["auc"], MacroF1_tuned_thr=m["macro_f1"], MacroF1_at_0_5=macro_f1_np(y_dev, (pr >= 0.5).astype(int)), thr=th_))
    agree = np.mean((tp >= thr_t) == (p_dev >= THR))
    log(f"\nTEACHER vs STUDENT on dev (thresholds tuned per model; agreement of hard predictions: {100*agree:.1f}%, prob. correlation {np.corrcoef(tp, p_dev)[0,1]:.3f})")
    log(pd.DataFrame(cmp_rows).round(4).to_string(index=False)); pd.DataFrame(cmp_rows).round(4).to_csv(OUT / f"teacher_vs_student_{RUN_TAG}.csv", index=False)
else:
    log("\n(no teacher dev probabilities available - teacher-vs-student table skipped)")

ABL = [("all windows (attention pooling)",     None,                                  "att"),
       ("all windows (mean of window logits)", None,                                  "winmean"),
       ("coverage subset K=6 (training policy)", lambda g, n: np.isin(g, pick(n, 6)), "att"),
       ("first window only",                   lambda g, n: g == 0,                   "att"),
       ("last window only",                    lambda g, n: g == n - 1,               "att"),
       ("last 2 windows",                      lambda g, n: g >= n - 2,               "att"),
       ("first + last 2 windows",              lambda g, n: (g == 0) | (g >= n - 2),  "att"),
       ("first half of document",              lambda g, n: g <= (n - 1) / 2,         "att"),
       ("second half of document",             lambda g, n: g >= (n - 1) / 2,         "att")]
rows = []
for name, keep, agg in ABL:
    p = p_dev if (keep is None and agg == "att") else doc_probs(head, f_dev, ptr_dev, g_dev, n_dev, keep=keep, agg=agg)
    thr_, f_t, f_05 = tune_threshold(y_dev, p)
    rows.append(dict(setting=name, AUC=roc_auc_score(y_dev, p), MacroF1_at_0_5=f_05, MacroF1_tuned_thr=f_t, thr=thr_))
abl_df = pd.DataFrame(rows).round(4)
log("\nSECTION / POOLING ABLATION (student; tuned-thr column is optimistic - tuned on dev itself)")
log(abl_df.to_string(index=False)); abl_df.to_csv(OUT / f"ablation_sections_{RUN_TAG}.csv", index=False)

_, att = doc_probs(head, f_dev, ptr_dev, g_dev, n_dev, collect_att=True)
mass = np.zeros(NB + 2); cnt_ = np.zeros(NB + 2)
for b, w in att:
    for bi, wi in zip(b, w): mass[bi] += wi; cnt_[bi] += 1
labels_b = ["first"] + [f"rel {i/NB:.2f}-{(i+1)/NB:.2f}" for i in range(NB)] + ["last"]
att_df = pd.DataFrame({"window position": labels_b, "mean attention weight": mass / np.maximum(cnt_, 1), "share of total attention": mass / mass.sum(), "n windows": cnt_.astype(int)}).round(4)
log("\nATTENTION BY WINDOW POSITION (dev, student)"); log(att_df.to_string(index=False)); att_df.to_csv(OUT / f"attention_by_position_{RUN_TAG}.csv", index=False)

pred_dev = (p_dev >= THR).astype(int); ntok = ds_dev.n_tok
grp = np.where((pred_dev == 1) & (y_dev == 1), "TP", np.where((pred_dev == 0) & (y_dev == 0), "TN", np.where((pred_dev == 1) & (y_dev == 0), "FP", "FN")))
err_df = pd.DataFrame({"group": grp, "tokens": ntok, "windows": n_dev, "p1": p_dev, "mixed_case_src": ds_dev.has_upper})
summ = err_df.groupby("group").agg(n=("tokens", "size"), mean_tokens=("tokens", "mean"), median_tokens=("tokens", "median"), mean_windows=("windows", "mean"), mean_p1=("p1", "mean"), mixed_case_share=("mixed_case_src", "mean")).round(2)
log("\nERROR ANALYSIS at tuned threshold (FP = predicted 1, true 0 | FN = predicted 0, true 1)"); log(summ.to_string())
q = pd.qcut(ntok, 4, duplicates="drop")
er = pd.DataFrame({"len_quartile": q, "wrong": pred_dev != y_dev, "y": y_dev}).groupby("len_quartile", observed=True).agg(docs=("wrong", "size"), error_rate=("wrong", "mean"), class1_share=("y", "mean")).round(3)
log("\nERROR RATE BY DOCUMENT LENGTH QUARTILE (tokens)"); log(er.to_string())
summ.to_csv(OUT / f"error_groups_{RUN_TAG}.csv"); er.to_csv(OUT / f"error_by_length_{RUN_TAG}.csv")
pd.DataFrame({"id": ds_dev.ids, "label": y_dev, "p1": p_dev, "pred": pred_dev, "tokens": ntok, "windows": n_dev,
              **({"p1_teacher": TEACHER_DEV_P} if TEACHER_DEV_P is not None else {})}).to_csv(OUT / f"dev_probs_{RUN_TAG}.csv", index=False)

n_head = sum(p.numel() for p in head.parameters())
eff = dict(student_backbone_frozen_params=n_total - n_train, student_lorma_trainable_params=n_train, student_head_params=n_head,
           student_trainable_pct=100 * n_train / n_total, adapter_head_ckpt_MB=(os.path.getsize(BEST) / 2**20 if BEST.exists() else None),
           student_backbone_fp16_MB=2 * (n_total - n_train) / 2**20, peak_gpu_GB=(torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else None),
           student_dev_windows=int(len(f_dev)), student_windows_per_s=len(f_dev) / max(dev_sec, 1e-6), tokenizer=TOK.name,
           teacher_params=T_META.get("t_params"), teacher_dev_windows_per_s=T_META.get("t_dev_wps"))
if eff["teacher_params"]: eff["param_ratio_teacher_over_student"] = eff["teacher_params"] / (n_total - n_train)
if eff["teacher_dev_windows_per_s"]: eff["speed_ratio_student_over_teacher"] = eff["student_windows_per_s"] / eff["teacher_dev_windows_per_s"]
log("\nEFFICIENCY: " + json.dumps(eff, default=float))
json.dump({"threshold": THR, "dev_at_0.5": M05, "dev_at_tuned": MT, "teacher_vs_student": cmp_rows, "run": RUN_CFG, "train_state": S, "efficiency": eff},
          open(OUT / f"dev_metrics_{RUN_TAG}.json", "w"), indent=1, default=float)

evaluating best adapter from /kaggle/working/phase4_m4_kd/ckpt/best_adapter_m4kd.pt (step 3025, quick AUC 0.9130)
    [dev] batch 101/347 | 88 windows/s (avg)
    [dev] batch 201/347 | 88 windows/s (avg)
    [dev] batch 301/347 | 87 windows/s (avg)
    [dev] batch 347/347 | 88 windows/s (avg)
dev feature extraction (student): 2.1 min

STUDENT DEV RESULTS (1,988 docs, class balance [994, 994])
threshold 0.50  | Macro-F1 0.8045 | Macro-P 0.8068 | Macro-R 0.8048 | Acc 0.8048 | AUC 0.8955
                | class0 P/R/F1 0.782/0.845/0.812 | class1 P/R/F1 0.832/0.765/0.797 | confusion [[TN,FP],[FN,TP]] = [[840, 154], [234, 760]]
tuned thr 0.40  | Macro-F1 0.8129 | Macro-P 0.8129 | Macro-R 0.8129 | Acc 0.8129 | AUC 0.8955
                | class0 P/R/F1 0.809/0.819/0.814 | class1 P/R/F1 0.817/0.807/0.812 | confusion [[TN,FP],[FN,TP]] = [[814, 180], [192, 802]]

TEACHER vs STUDENT on dev (thresholds tuned per model; agreement of hard predictions: 87.9%, prob. correlation 0.855)
               

In [10]:
t0 = time.time()
test_idx = np.arange(len(ds_test))
f_te, ptr_te, g_te, n_te = extract_features(model, ds_test, test_idx, MAX_INFER_WINDOWS, tag="test")
log(f"test feature extraction: {(time.time()-t0)/60:.1f} min")
np.savez(OUT / f"test_window_feats_{RUN_TAG}.npz", feats=f_te, ptr=ptr_te, gidx=g_te, nall=n_te)
p_te = doc_probs(head, f_te, ptr_te, g_te, n_te)
pred_te = (p_te >= THR).astype(int)

pd.DataFrame({"id": ds_test.ids, "p1": p_te, "pred": pred_te}).to_csv(OUT / f"test_probs_{RUN_TAG}.csv", index=False)      # probabilities kept for ensembling
submission = [{"id": i, "pred": int(v)} for i, v in zip(ds_test.ids, pred_te)]

assert len(submission) == len(ds_test) == len(set(ds_test.ids)), "one prediction per test case required"
assert all(set(s) == {"id", "pred"} and type(s["pred"]) is int and s["pred"] in (0, 1) for s in submission)
if not SMOKE:
    ref = [str(r["id"]) for r in _records("test")]
    assert [s["id"] for s in submission] == ref, "submission ids/order differ from test.json"
sub_path = SUB / "prediction.json"
with open(sub_path, "w", encoding="utf-8") as f: json.dump(submission, f)
with zipfile.ZipFile(SUB / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(sub_path, "prediction.json")

log("\n" + "=" * 90)
log(f"SUBMISSION WRITTEN: {sub_path}   ({len(submission):,} predictions{' - SMOKE RUN, NOT A VALID SUBMISSION' if SMOKE else ''})")
log(f"predicted class balance: class0={int((pred_te==0).sum()):,} ({100*(pred_te==0).mean():.1f}%) | class1={int((pred_te==1).sum()):,} ({100*(pred_te==1).mean():.1f}%) | threshold {THR:.2f}")
log(f"student dev Macro-F1 (leak-free dev, tuned thr): {MT['macro_f1']:.4f} | at 0.5: {M05['macro_f1']:.4f} | AUC {MT['auc']:.4f}")
log(f"total wall time: {(time.time()-T0)/60:.0f} min | outputs in {WORK}")
log("=" * 90)

    [test] batch 101/272 | 87 windows/s (avg)
    [test] batch 201/272 | 87 windows/s (avg)
    [test] batch 272/272 | 88 windows/s (avg)
test feature extraction: 1.7 min


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            


SUBMISSION WRITTEN: /kaggle/working/phase4_m4_kd/submission/prediction.json   (1,517 predictions)
predicted class balance: class0=690 (45.5%) | class1=827 (54.5%) | threshold 0.40
student dev Macro-F1 (leak-free dev, tuned thr): 0.8129 | at 0.5: 0.8045 | AUC 0.8955
total wall time: 240 min | outputs in /kaggle/working/phase4_m4_kd
